In [17]:
import cv2
import tensorflow as tf
import numpy as np
import json
from collections import deque
import random
import re
import pytesseract
from dotenv import load_dotenv
from llama_cpp import Llama
from supertonic import TTS

load_dotenv()

parametros_file = open('dataset/parameters.jsonc', 'r', encoding='utf-8')
parametros = json.load(parametros_file)

models = {
    'health': tf.keras.models.load_model('./models/health.keras'),
    'ultimate': tf.keras.models.load_model('./models/ultimate.keras')
}

# Módulo de visión

Identificar nivel de vida y estado de hábilidad ultimate

In [18]:
video = "test1.mp4"
fps = 60
procesar_cada_n_frames = 60
comentar_cada_n_frames = 10 * 60  # segundos * frames de video
historial_deteccion = []


def recortar_imagen_categoria(categoria: str, image: np.ndarray) -> np.ndarray:
    crops = []
    for section in parametros[categoria]['sections']:
        left = section['x']
        top = section['y']
        right = section['x'] + section['width']
        bottom = section['y'] + section['height']
        crops.append(image[top:bottom, left:right])

    max_height = max(img.shape[0] for img in crops)
    padded_crops = []

    for img in crops:
        if img.shape[0] < max_height:
            padding = np.zeros((max_height - img.shape[0], img.shape[1], img.shape[2]), dtype=img.dtype)
            img = np.concatenate([img, padding], axis=0)
        padded_crops.append(img)

    return np.concatenate(padded_crops, axis=1)


def preprocess_health_cv(image_np):
    image_rgb = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
    image_float = image_rgb.astype(np.float32) / 255.0
    grayscale = cv2.cvtColor(image_float, cv2.COLOR_RGB2GRAY)
    grayscale_3ch = cv2.cvtColor(grayscale, cv2.COLOR_GRAY2RGB)
    return grayscale_3ch


splatted_rectangles = [
    {'x': 569, 'y': 664, 'width': 78, 'height': 28},
    {'x': 569, 'y': 619, 'width': 78, 'height': 28},
    {'x': 569, 'y': 577, 'width': 78, 'height': 28},
    {'x': 569, 'y': 532, 'width': 78, 'height': 28},
]


def recortar_zonas_splatted(image: np.ndarray) -> list[np.ndarray]:
    crops = []
    for rectangle in splatted_rectangles:
        left = rectangle['x']
        top = rectangle['y']
        right = left + rectangle['width']
        bottom = top + rectangle['height']
        crops.append(image[top:bottom, left:right])
    return crops


def preprocess_splatted_ocr(image: np.ndarray) -> np.ndarray:
    grayscale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    grayscale = cv2.resize(grayscale, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    grayscale = cv2.GaussianBlur(grayscale, (3, 3), 0)
    _, threshold = cv2.threshold(grayscale, 180, 255, cv2.THRESH_BINARY)
    return threshold


def contar_splatted_ocr(frame_rgb: np.ndarray) -> int:
    splatted_count = 0
    for zona_splatted in recortar_zonas_splatted(frame_rgb):
        imagen_ocr = preprocess_splatted_ocr(zona_splatted)
        texto = pytesseract.image_to_string(
            imagen_ocr,
            config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz",
        )
        if re.search(r"\bsplatted\b", texto, flags=re.IGNORECASE):
            splatted_count += 1
    return splatted_count


respawn_rectangle = {'x': 1094, 'y': 648, 'width': 88, 'height': 29}


def recortar_zona_respawn(image: np.ndarray) -> np.ndarray:
    left = respawn_rectangle['x']
    top = respawn_rectangle['y']
    right = left + respawn_rectangle['width']
    bottom = top + respawn_rectangle['height']
    return image[top:bottom, left:right]


def preprocess_respawn_ocr(image: np.ndarray) -> np.ndarray:
    grayscale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    grayscale = cv2.resize(grayscale, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    grayscale = cv2.GaussianBlur(grayscale, (3, 3), 0)
    _, threshold = cv2.threshold(grayscale, 170, 255, cv2.THRESH_BINARY)
    return threshold


def detectar_respawn_ocr(frame_rgb: np.ndarray) -> bool:
    zona_respawn = recortar_zona_respawn(frame_rgb)
    imagen_ocr = preprocess_respawn_ocr(zona_respawn)
    texto = pytesseract.image_to_string(
        imagen_ocr,
        config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz",
    )
    return re.search(r"\brespawn\b", texto, flags=re.IGNORECASE) is not None


timer_rectangle = {'x': 600, 'y': 31, 'width': 74, 'height': 37}


def preprocess_timer_ocr(image: np.ndarray) -> np.ndarray:
    grayscale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    grayscale = cv2.resize(grayscale, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    grayscale = cv2.GaussianBlur(grayscale, (3, 3), 0)
    _, threshold = cv2.threshold(grayscale, 160, 255, cv2.THRESH_BINARY)
    return threshold


def leer_tiempo_ocr(frame_rgb: np.ndarray) -> int | None:
    left = timer_rectangle['x']
    top = timer_rectangle['y']
    right = left + timer_rectangle['width']
    bottom = top + timer_rectangle['height']
    zona_timer = frame_rgb[top:bottom, left:right]
    imagen_ocr = preprocess_timer_ocr(zona_timer)
    texto = pytesseract.image_to_string(
        imagen_ocr,
        config="--psm 7 -c tessedit_char_whitelist=0123456789:",
    )
    match = re.search(r"(\d{1,2})\s*:?\s*(\d{2})", texto)
    if not match:
        return None
    minutos = int(match.group(1))
    segundos = int(match.group(2))
    if segundos >= 60:
        return None
    return minutos * 60 + segundos


def obtener_game_status(tiempo_transcurrido: float | None, frame: int) -> str:
    segundos = tiempo_transcurrido if tiempo_transcurrido is not None else frame / fps
    return 'start' if segundos < 15 else 'mid game' if segundos < 120 else 'last minute' if segundos < 150 else 'last 30 seconds'


# --- VARIABLES DE ESTADO Y DEBOUNCE CORREGIDAS ---
niveles_health = [3, 1, 2, 0]

frame_actual = 0
frame_ultimo_comentario = 0
frame_ultimo_splat = -10 ** 9

# Cooldowns para evitar solapamientos constantes
cooldown_splat_frames = 5 * fps
cooldown_general_frames = 4 * fps  # Esperar mínimo 4 segs entre comentarios de estado

tiempo_inicio_ocr = None

historial_salud_reciente = deque(maxlen=3)
estado_salud_estable = 0 # DEBE INICIAR EN 0 (Salud Completa/OK)
historial_ultimate_reciente = deque(maxlen=3)
estado_ultimate_estable = 'not_ready'

cap = cv2.VideoCapture(video)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_actual % procesar_cada_n_frames != 0:
        frame_actual += 1
        continue

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    tiempo_ocr = leer_tiempo_ocr(frame_rgb)
    if tiempo_inicio_ocr is None and tiempo_ocr is not None:
        tiempo_inicio_ocr = tiempo_ocr

    tiempo_transcurrido_ocr = None
    if tiempo_inicio_ocr is not None and tiempo_ocr is not None:
        tiempo_transcurrido_ocr = max(0, tiempo_inicio_ocr - tiempo_ocr)

    resultados = {
        'frame': frame_actual,
        'tiempo_ocr': tiempo_ocr,
        'tiempo_transcurrido_ocr': tiempo_transcurrido_ocr,
        'debe_comentar': False,
        'health_worsened': False,
        'health_recovered': False,
        'ultimate_became_ready': False,
        'ultimate_used': False,
        'splatted_count': 0,
        'splatted_detected': False,
        'player_was_killed': False,
        'tempo_comment': False,
        'game_status': obtener_game_status(tiempo_transcurrido_ocr, frame_actual)
    }

    for categoria in parametros:
        frame_categoria = recortar_imagen_categoria(categoria, frame_rgb)
        frame_categoria = cv2.resize(frame_categoria, (228, 228), interpolation=cv2.INTER_AREA)

        if categoria == "health":
            frame_categoria = preprocess_health_cv(frame_categoria)

        predicciones = models[categoria].predict(np.expand_dims(frame_categoria, axis=0), verbose=0)
        class_idx = np.argmax(predicciones)
        label = parametros[categoria]['labels'][class_idx]
        resultados[categoria] = label
        resultados[categoria + '_idx'] = class_idx.item()

    splatted_count = contar_splatted_ocr(frame_rgb)
    resultados['splatted_count'] = splatted_count
    if splatted_count > 0:
        resultados['splatted_detected'] = True

    if detectar_respawn_ocr(frame_rgb):
        resultados['player_was_killed'] = True

    # --- LÓGICA DE DEBOUNCE (ANTI-PARPADEO) ---
    if 'health_idx' in resultados:
        historial_salud_reciente.append(niveles_health[resultados['health_idx']])
    if 'ultimate' in resultados:
        historial_ultimate_reciente.append(resultados['ultimate'])

    if len(historial_salud_reciente) == 3 and len(set(historial_salud_reciente)) == 1:
        nuevo_estado_salud = historial_salud_reciente[0]
    else:
        nuevo_estado_salud = estado_salud_estable

    if len(historial_ultimate_reciente) == 3 and len(set(historial_ultimate_reciente)) == 1:
        nuevo_estado_ultimate = historial_ultimate_reciente[0]
    else:
        nuevo_estado_ultimate = estado_ultimate_estable

    # Registrar cambios significativos
    flag_evento_prioritario = False

    if len(historial_deteccion) >= 1:
        if nuevo_estado_salud > estado_salud_estable and nuevo_estado_salud >= 2:
            resultados['health_worsened'] = True
            flag_evento_prioritario = True
        elif nuevo_estado_salud < estado_salud_estable and estado_salud_estable >= 2:
            resultados['health_recovered'] = True
            flag_evento_prioritario = True

        if nuevo_estado_ultimate == 'ready' and estado_ultimate_estable != 'ready':
            resultados['ultimate_became_ready'] = True
            flag_evento_prioritario = True
        elif nuevo_estado_ultimate != 'ready' and estado_ultimate_estable == 'ready':
            resultados['ultimate_used'] = True
            flag_evento_prioritario = True

    estado_salud_estable = nuevo_estado_salud
    estado_ultimate_estable = nuevo_estado_ultimate

    # Lógica de Tempo
    flag_tempo = False
    if frame_actual - frame_ultimo_comentario >= comentar_cada_n_frames:
        resultados['tempo_comment'] = True
        flag_tempo = True

    # --- PUERTA DE CONTROL DE PUBLICACIÓN (COOLDOWNS) ---
    frames_desde_ultimo = frame_actual - frame_ultimo_comentario
    debe_publicar = False

    if resultados['splatted_detected'] or resultados['player_was_killed']:
        # Las muertes interrumpen casi todo
        if frames_desde_ultimo >= cooldown_splat_frames:
            debe_publicar = True
            frame_ultimo_splat = frame_actual
    elif flag_evento_prioritario:
        # Vida y Especiales esperan al menos 4 segundos para no encimarse
        if frames_desde_ultimo >= cooldown_general_frames:
            debe_publicar = True
    elif flag_tempo:
        # Los comentarios de relleno respetan sus 10 segundos
        debe_publicar = True

    if debe_publicar:
        frame_ultimo_comentario = frame_actual
        resultados['debe_comentar'] = True

    frame_actual += 1
    historial_deteccion.append(resultados.copy())

cap.release()

# Módulo de narración

In [19]:
llm_text_gen = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-E4B-it-GGUF",
    filename="gemma-4-E4B-it-Q8_0.gguf",
    n_ctx=8192,
    n_gpu_layers=-1,
)

/Users/maple/Developer/inteligencia-artificial/narrador-splatoon/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_model_loader: loaded meta data with 56 key-value pairs and 720 tensors from /Users/maple/.cache/huggingface/hub/models--unsloth--gemma-4-E4B-it-GGUF/snapshots/653803f092503c04a65164346f3208a36e707693/./gemma-4-E4B-it-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma4
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3

In [20]:
comentarios = []


def traducir_contexto(resultados, eventos, splatted_count):
    tiempo = resultados.get('tiempo_ocr')
    estado_juego = resultados.get('game_status', 'juego')

    traduccion = f"La partida está en la fase de {estado_juego}. "
    if tiempo is not None and tiempo < 60:
        traduccion += f"Quedan solo {tiempo} segundos en el reloj. "

    if splatted_count > 0:
        traduccion += f"¡El jugador acaba de conseguir {splatted_count} baja(s) rápida(s) (splat)! Es una jugada clave. "
    elif resultados.get('player_was_killed'):
        traduccion += "Mataron al jugador. "
    else:
        # Ahora confiamos solo en los eventos confirmados, no en la lectura del frame suelto
        if 'health_worsened' in eventos:
            traduccion += "El jugador ha recibido muchísimo daño de repente, su vida está en peligro. "
        elif 'health_recovered' in eventos:
            traduccion += "El jugador logró recuperarse del daño y su salud está estable de nuevo. "

        if 'ultimate_became_ready' in eventos:
            traduccion += "La habilidad especial acaba de terminar de cargar y está lista para usarse. "
        elif 'ultimate_used' in eventos:
            traduccion += "El jugador acaba de activar su habilidad especial para presionar. "

    if not any(e in eventos for e in ['splatted_detected', 'health_worsened', 'health_recovered', 'ultimate_became_ready', 'ultimate_used', 'player_was_killed']):
        contextos_inactivos = [
            "El jugador está relajado, pintando el mapa con alegría y fluyendo por el escenario.",
            "Hay muchísima tensión y paranoia en el aire, el jugador pinta esperando una emboscada inminente.",
            "El jugador está metiendo presión agresiva, asfixiando al rival con pintura para arrinconarlo.",
            "Movimiento frenético puro: el jugador rota y pinta a toda velocidad sin quedarse quieto.",
            "El jugador está aplicando una estrategia paciente, buscando posiciones seguras con inteligencia."
        ]
        traduccion += random.choice(contextos_inactivos)

    return traduccion


def comentario_fallback_dinamico(eventos, comment_history):
    aperturas = ["¡Atención!", "¡Cuidado!", "¡Ojo aquí!", "¡Qué momento!", "¡Se mueve la partida!", "¡Atentos!"]

    if 'splatted_detected' in eventos:
        bases = ["Bajas encadenadas, toca presionar la zona.", "Splats a favor, se abre el mapa.",
                 "¡Varios menos en el equipo rival!"]
    elif 'player_was_killed' in eventos:
        bases = ["Cayó el jugador, toca pensar la reentrada.", "Reaparición en marcha, hay que reagruparse.",
                 "Lo liquidan y ahora toca volver con calma."]
    elif 'health_worsened' in eventos:
        bases = ["Va muy tocado, tiene que esconderse.", "La tinta enemiga aprieta, necesita cobertura.",
                 "Está al borde de caer, ¡cuidado!"]
    elif 'health_recovered' in eventos:
        bases = ["Recupera vida y puede volver a pelear.", "Ya estabilizó la salud, toca retomar espacio.",
                 "Sale del peligro y vuelve a pintar con calma."]
    elif 'ultimate_became_ready' in eventos:
        bases = ["Especial cargado y listo para soltarlo.", "Ojo con el especial que puede cambiar todo.",
                 "Tiene la ulti lista en la bolsa."]
    elif 'ultimate_used' in eventos:
        bases = ["Especial en camino, toca ganar terreno.", "Suelta el especial para romper la línea.",
                 "¡Aprovechando esa ulti para presionar!"]
    else:
        bases = [
            # Alegría y Diversión (Disfrutando el juego)
            "¡Qué bonito es ver el mapa llenarse de color a este ritmo!",
            "Pintando por aquí, pintura por allá, ¡dejando todo impecable!",
            "¡Navegando por la tinta con todo el estilo del mundo!",
            "Pintando de lo lindo y pasándola genial en el mapa.",
            "¡Haciendo verdadero arte en el escenario a base de tinta!",
            "Disfrutando de la partida y cubriendo cada rincón con alegría.",

            # Tranquilidad y Relax (Tomando un respiro)
            "Un momento de paz para nadar tranquilo y recargar el tanque.",
            "Respirando profundo y pintando el mapa sin nada de estrés.",
            "Todo fluye con calma, asegurando el terreno como en un paseo.",
            "Relajando el ritmo un segundito para disfrutar del entintado.",
            "Pintando a sus anchas, sin nadie que moleste por ahora.",
            "Aprovechando la calma para dejar la casa limpia y pintada.",

            # Diversión táctica (Juego fresco)
            "Un poco de limpieza en el mapa nunca viene mal, ¡a pintar se ha dicho!",
            "Moviéndose como pez en el agua, preparando la zona con mucha frescura.",
            "¡Saltando y pintando a gusto mientras preparan la siguiente jugada!",
            "Cubriendo el mapa con mucha buena vibra y excelente control.",
            "¡Fluyendo por la tinta como si estuvieran surfeando en la zona!",

            # Tensión y Paranoia (Esperando el ataque)
            "¡El ambiente se corta con tijera! Todos buscando el mínimo error.",
            "¡Tensión al máximo! Nadie cede un solo milímetro de tinta.",
            "¡Mucho cuidado con los flancos, en cualquier momento salta la chispa!",
            "Ese silencio en el mapa es peligroso, alguien está preparando una emboscada.",
            "¡Pintando con nervios de acero, sabiendo que el rival acecha en la tinta!",

            # Agresividad contenida y Presión (Dominando el mapa)
            "¡Pintando con furia para arrinconar y asfixiar al equipo contrario!",
            "¡Metiendo presión pura a base de pintura, ahogando las salidas del rival!",
            "¡Ese entintado es una amenaza directa, se viene el empuje fuerte!",
            "¡Imponiendo respeto en el centro, obligando al rival a retroceder!",
            "¡Devorando el mapa! Quieren dejar al enemigo sin opciones de movilidad.",

            # Velocidad y Movimiento frenético
            "¡Movimiento frenético en la zona, no se quedan quietos ni un segundo!",
            "¡Qué agilidad para rotar y seguir pintando el mapa sin parar!",
            "¡Velocidad a tope! Cubriendo cada rincón antes de que el rival reaccione.",
            "¡No hay tiempo que perder, entintando a una velocidad de locura!",
            "¡Rotaciones rapidísimas para mantener al enemigo desorientado!",

            # Estrategia agresiva
            "Cerrando todas las vías de escape, preparan la trampa perfecta.",
            "¡Asegurando la altura de forma agresiva para dominar la visión!",
            "Pintando de forma estratégica, ¡quieren encerrarlos en su propia base!",
            "¡Forzando al rival a jugar incómodo robándole todo el territorio!",
            "Preparando el terreno para la siguiente masacre, ¡pura estrategia!"
        ]

    random.shuffle(aperturas)
    random.shuffle(bases)

    for apertura in aperturas:
        for base in bases:
            candidato = f"{apertura} {base}"
            if not es_demasiado_repetido(candidato, comment_history):
                return candidato

    # Si por alguna razón todo falla, devuelve uno seguro
    return "¡Hay que mantener el ritmo en la zona!"


def normalizar_comentario(texto: str) -> str:
    texto = texto.strip()
    texto = re.sub(r"^['\"“”‘’]+|['\"“”‘’]+$", "", texto)
    texto = texto.split("\n", 1)[0].strip()
    if len(texto) > 120:
        texto = texto[:120].rsplit(" ", 1)[0]
    return texto


def es_demasiado_repetido(texto: str, comment_history) -> bool:
    texto_normalizado = texto.lower().strip("¡!., ")
    for previo in comment_history:
        previo_normalizado = previo.lower().strip("¡!., ")
        if texto_normalizado == previo_normalizado:
            return True
        palabras = set(texto_normalizado.split())
        palabras_previas = set(previo_normalizado.split())
        if palabras and len(palabras & palabras_previas) / len(palabras) >= 0.75:
            return True
    return False


def comentario_incompleto(texto: str) -> bool:
    texto = texto.strip()
    if not texto:
        return True
    if texto.endswith((",", "...", "…", ":", ";", " y", " pero", " con", " para")):
        return True
    return texto[-1] not in ".!?¡!"


def resumir_evento(resultados):
    eventos = []
    if resultados.get('splatted_count', 0) > 0:
        eventos.append('splatted_detected')
    if resultados.get('player_was_killed'):
        eventos.append('player_was_killed')
    if resultados.get('tempo_comment'):
        eventos.append('tempo_comment')
    if resultados.get('ultimate_became_ready'):
        eventos.append('ultimate_became_ready')
    if resultados.get('ultimate_used'):
        eventos.append('ultimate_used')
    if resultados.get('health_worsened'):
        eventos.append('health_worsened')
    if resultados.get('health_recovered'):
        eventos.append('health_recovered')
    if not eventos:
        eventos.append('tempo_comment')
    return eventos


frames_splat_agrupados = set()
ventana_agrupacion_splats_frames = 5 * fps

for i in range(len(historial_deteccion)):
    resultados = historial_deteccion[i]
    if resultados['frame'] in frames_splat_agrupados:
        continue
    if not resultados['debe_comentar']:
        continue

    eventos = resumir_evento(resultados)
    splatted_count = resultados.get('splatted_count', 0)

    if resultados.get('splatted_detected'):
        frame_splat_inicial = resultados['frame']
        splat_frames_unicos = [frame_splat_inicial]
        splatted_count = max(1, splatted_count)
        for futuro in historial_deteccion[i + 1:]:
            if futuro['frame'] - frame_splat_inicial > ventana_agrupacion_splats_frames:
                break
            if not futuro.get('splatted_detected'):
                break
            frames_splat_agrupados.add(futuro['frame'])
            futuro_splatted_count = max(1, futuro.get('splatted_count', 1))
            if futuro_splatted_count > splatted_count:
                splat_frames_unicos.append(futuro['frame'])
                splatted_count = futuro_splatted_count
        resultados = resultados.copy()
        resultados['splatted_count'] = splatted_count
        resultados['splatted_grouped_count'] = splatted_count
        resultados['splatted_grouped_frames'] = splat_frames_unicos

    # Traducimos todo el caos de variables a un párrafo simple para el LLM
    contexto_natural = traducir_contexto(resultados, eventos, splatted_count)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un caster (comentarista) profesional de e-sports especializado en Splatoon. "
                "Tu objetivo es narrar la acción en tiempo real con alta energía en UNA SOLA FRASE.\n\n"

                "LO QUE ESTÁ PASANDO EN ESTE INSTANTE:\n"
                f"{contexto_natural}\n\n"

                "REGLAS ESTRICTAS DE JERGA Y TONO:\n"
                "- Usa terminología de Splatoon (Splat, liquidado, entintar, zona, especial, holdear, pushear).\n"
                "- No inventes nombres de equipos ni marcadores exactos.\n"
                "- NO uses frases genéricas como 'el control está en sus manos'.\n"
                "- Si el evento es respawn, habla de la caída y la reentrada, no de un splat a favor.\n"
                "- Tu respuesta debe tener entre 40 y 85 caracteres.\n"
                "- Devuelve SOLO el comentario final, sin comillas, sin introducciones ('Aquí está...') ni emojis."
            )
        },
        {
            "role": "user",
            "content": "Narra lo que acaba de pasar en la partida."
        }
    ]

    historial_comentarios = [c['comentario'] for c in comentarios[-10:]]
    texto_generado = None

    for intento in range(3):
        salida = llm_text_gen.create_chat_completion(
            mensajes,
            max_tokens=48,
            temperature=0.75 + (intento * 0.1),  # Bajamos un poco la temperatura base para mayor coherencia
            repeat_penalty=1.12,  # Reducido de 1.20 a 1.12 para no romper la gramática del español
            stop=["\n", "\n\n", "\"", "Aquí"],  # Añadimos comillas para que corte si intenta citar algo
        )

        candidato = salida['choices'][0]['message']['content']
        candidato = normalizar_comentario(candidato)

        if not es_demasiado_repetido(candidato, historial_comentarios) and not comentario_incompleto(candidato):
            texto_generado = candidato
            break

    # Si el LLM falla 3 veces en dar un buen comentario, entra el Fallback Dinámico
    if texto_generado is None:
        texto_generado = comentario_fallback_dinamico(eventos, historial_comentarios)

    generacion = {
        "frame": resultados['frame'],
        "comentario": texto_generado,
        "splatted_detected": resultados.get('splatted_detected', False),
        "splatted_count": resultados.get('splatted_count', 0),
        "splatted_grouped_frames": resultados.get('splatted_grouped_frames', []),
        "player_was_killed": resultados.get('player_was_killed', False),
        "health_worsened": resultados.get('health_worsened', False),
        "health_recovered": resultados.get('health_recovered', False),
        "health": resultados.get('health'),
    }

    comentarios.append(generacion)
    print(f"[Frame {resultados['frame']}] {texto_generado}")

llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =     618.48 ms /   243 tokens (    2.55 ms per token,   392.90 tokens per second)
llama_perf_context_print:        eval time =    1370.65 ms /    23 runs   (   59.59 ms per token,    16.78 tokens per second)
llama_perf_context_print:       total time =    2062.40 ms /   266 tokens
llama_perf_context_print:    graphs reused =         21
Llama.generate: 63 prefix-match hit, remaining 189 prompt tokens to eval


[Frame 600] ¡Movimiento frenético! El jugador entinta cada rincón, pusheando sin descanso desde el inicio.


llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =     512.66 ms /   189 tokens (    2.71 ms per token,   368.67 tokens per second)
llama_perf_context_print:        eval time =    1207.72 ms /    21 runs   (   57.51 ms per token,    17.39 tokens per second)
llama_perf_context_print:       total time =    1750.72 ms /   210 tokens
llama_perf_context_print:    graphs reused =         19
Llama.generate: 66 prefix-match hit, remaining 165 prompt tokens to eval


[Frame 1200] ¡Qué buen splat! El aguante se rompió y el territorio queda entintado.


llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =     478.06 ms /   165 tokens (    2.90 ms per token,   345.15 tokens per second)
llama_perf_context_print:        eval time =    1311.31 ms /    20 runs   (   65.57 ms per token,    15.25 tokens per second)
llama_perf_context_print:       total time =    1833.06 ms /   185 tokens
llama_perf_context_print:    graphs reused =         19
Llama.generate: 66 prefix-match hit, remaining 181 prompt tokens to eval


...

[Frame 11160] ¡Rodando y entintando al máximo en estos últimos segundos, puro pusheo frenético por la zona!


llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =     427.03 ms /   178 tokens (    2.40 ms per token,   416.84 tokens per second)
llama_perf_context_print:        eval time =    1003.57 ms /    20 runs   (   50.18 ms per token,    19.93 tokens per second)
llama_perf_context_print:       total time =    1449.52 ms /   198 tokens
llama_perf_context_print:    graphs reused =         18
Llama.generate: 72 prefix-match hit, remaining 171 prompt tokens to eval


[Frame 11760] ¡Pura vibra! Entintando con flow total en estos últimos segundos de la zona.


llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =     442.64 ms /   171 tokens (    2.59 ms per token,   386.31 tokens per second)
llama_perf_context_print:        eval time =    1529.67 ms /    29 runs   (   52.75 ms per token,    18.96 tokens per second)
llama_perf_context_print:       total time =    2018.44 ms /   200 tokens
llama_perf_context_print:    graphs reused =         27
Llama.generate: 242 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     623.59 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    1039.51 ms /    21 runs   (   49.50 ms per token,    20.20 tokens per second)
llama_perf_context_print:       total time =    1060.66 ms /    22 tokens
llama_perf_context_print:    graphs reused =         19


[Frame 12360] Aguanta el *hold* con inteligencia, buscando la entrada perfecta para ese último pusheo.


# Módulo de narración (voz)

In [21]:
tts = TTS(auto_download=True)
style = tts.get_voice_style(voice_name="M1")

In [22]:
for comentario in comentarios:
    wav, duration = tts.synthesize(comentario['comentario'], voice_style=style, speed=1.25)
    comentario['duration'] = duration
    tts.save_audio(wav, f"tts/{comentario['frame']}.wav")

# Generación de video (ffmpeg)

In [23]:
# Generación de subtitulos

def segundos_a_srt(segundos: float) -> str:
    total_ms = max(0, round(segundos * 1000))
    horas = total_ms // 3_600_000
    total_ms %= 3_600_000
    minutos = total_ms // 60_000
    total_ms %= 60_000
    segundos_enteros = total_ms // 1000
    milisegundos = total_ms % 1000
    return f"{horas:02d}:{minutos:02d}:{segundos_enteros:02d},{milisegundos:03d}"

srt_data = ""
for srt_i, comentario in enumerate(comentarios, start=1):
    proximo_comentario = comentarios[srt_i] if srt_i < len(comentarios) else None
    inicio = comentario['frame'] / fps
    fin = (proximo_comentario['frame'] / fps) if proximo_comentario is not None else (frame_actual / fps)

    srt_data += f"{srt_i}\n{segundos_a_srt(inicio)} --> {segundos_a_srt(fin)}\n{comentario['comentario']}\n\n"

with open("comentarios.srt", "w", encoding="utf-8") as srt_file:
    srt_file.write(srt_data)



In [24]:
import platform
import subprocess

cmd = ["ffmpeg", "-y", "-i", video, "-i", "comentarios.srt"]
video_codec = "h264_videotoolbox" if platform.system() == "Darwin" and platform.machine() == "arm64" else "libx264"

for comentario in comentarios:
    cmd.extend(["-i", f"tts/{comentario['frame']}.wav"])

filter_parts = []
filter_parts.append("[0:a]volume=0.5[main_a]")
audio_inputs = ["[main_a]"]

# Obtenemos TODOS los tiempos de inicio para poder cortar los audios dinámicamente
all_start_seconds = [c['frame'] / fps for c in comentarios]

tts_input_offset = 2  # 0=video, 1=subtitulos, 2..=clips TTS
for i, comentario in enumerate(comentarios):
    start_seconds = all_start_seconds[i]
    duration = comentario.get('duration')

    # Buscamos cuándo empieza el SIGUIENTE comentario (si existe)
    next_start = all_start_seconds[i + 1] if i < len(comentarios) - 1 else None

    clip_filters = []

    # Si el siguiente comentario empieza ANTES de que este termine, lo cortamos (atrim)
    if next_start is not None and (duration is None or next_start < start_seconds + duration):
        trim_duration = next_start - start_seconds
        clip_filters.append(f"atrim=duration={trim_duration:.3f}")
        clip_filters.append("asetpts=PTS-STARTPTS")

    delay_ms = round(start_seconds * 1000)
    clip_filters.append(f"adelay={delay_ms}:all=1")

    # Volumen al 150% para que resalte sobre el juego
    clip_filters.append("volume=1.5")

    audio_label = f"a{i + 1}"
    filter_parts.append(f"[{i + tts_input_offset}:a]{','.join(clip_filters)}[{audio_label}]")
    audio_inputs.append(f"[{audio_label}]")

filter_parts.append(
    f"{''.join(audio_inputs)}amix=inputs={len(audio_inputs)}:duration=first:dropout_transition=0:normalize=0[outa]"
)

cmd.extend([
    "-filter_complex", ";".join(filter_parts),
    "-map", "0:v",
    "-map", "[outa]",
    "-map", "1:s",
    "-c:v", video_codec,
    "-c:a", "aac",
    "-c:s", "mov_text",
    "-shortest",
    f"comentarista_{video}"
])

print("Generando video final con audio ajustado y subtítulos...")
subprocess.run(cmd, check=True)



Generando video final con audio ajustado y subtítulos...


ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
[in#0 @ 0xb3b040000] UDTA parsing failed retrying raw
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'test5.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 1
    compatible

CompletedProcess(args=['ffmpeg', '-y', '-i', 'test5.mp4', '-i', 'comentarios.srt', '-i', 'tts/600.wav', '-i', 'tts/1200.wav', '-i', 'tts/1620.wav', '-i', 'tts/2220.wav', '-i', 'tts/2820.wav', '-i', 'tts/3420.wav', '-i', 'tts/3840.wav', '-i', 'tts/4440.wav', '-i', 'tts/5040.wav', '-i', 'tts/5340.wav', '-i', 'tts/5940.wav', '-i', 'tts/6420.wav', '-i', 'tts/7020.wav', '-i', 'tts/7560.wav', '-i', 'tts/8160.wav', '-i', 'tts/8760.wav', '-i', 'tts/9360.wav', '-i', 'tts/9660.wav', '-i', 'tts/10560.wav', '-i', 'tts/11160.wav', '-i', 'tts/11760.wav', '-i', 'tts/12360.wav', '-filter_complex', '[0:a]volume=0.5[main_a];[2:a]adelay=10000:all=1,volume=1.5[a1];[3:a]adelay=20000:all=1,volume=1.5[a2];[4:a]adelay=27000:all=1,volume=1.5[a3];[5:a]adelay=37000:all=1,volume=1.5[a4];[6:a]adelay=47000:all=1,volume=1.5[a5];[7:a]adelay=57000:all=1,volume=1.5[a6];[8:a]adelay=64000:all=1,volume=1.5[a7];[9:a]adelay=74000:all=1,volume=1.5[a8];[10:a]atrim=duration=5.000,asetpts=PTS-STARTPTS,adelay=84000:all=1,volume=